In [1]:
!apt-get update -y
!apt-get install -y proj-bin proj-data

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,592 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,864 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,637 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,600 kB]
Get:14 http://sec

In [2]:
!pip install geopandas
!pip install folium
!pip install shapely

In [8]:
!apt-get update -y
!apt-get install -y proj-bin proj-data

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 2s (2,432 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
proj-bin is already the newest version (9.3.1-1~ja

In [7]:
import pandas as pd
import geopandas as gpd
import folium
from shapely.geometry import Polygon
from scipy.spatial import Voronoi
import numpy as np


In [ ]:
#load school data file
school_locations_path = '/content/schools-list-calderdale.csv'
init_df = pd.read_csv(school_locations_path)

##APPLIED TO PRIMARY SCHOOLS - ALTER FOR DESIRED SCHOOLS/SEEDS. ENSURE FINAL DATAFRAME TO USE IS LABELLED AS df
#apply to primary schools only
df = init_df[init_df['Phase']=='Primary']

print (df)

In [20]:
#create dataframe with Point geometries - longitude and latitude
#use GeoDataFrame : has a geometry column for locaton data. Stores type of geometrc object i.e. pont and coordinates of object
##crs - 'coordinate refernce system'. EPSG -  'European Petroleum Survey Group'. '4326' - code for GPS Global Positioning System

##GPS projects the latitude and longitude coordinates from 3D planet onto flat surface of map.

gdf = gpd.GeoDataFrame(df, geometry = gpd.points_from_xy(df['Longitude'], df['Latitude']), crs = 'EPSG:4326')

     DfE number                             Establishment    Phase  \
0          2005                        Abbey Park Academy  Primary   
1          5201           All Saints' CofE Primary School  Primary   
2          2093        Ash Green Community Primary School  Primary   
3          2040  Bailiffe Bridge Junior and Infant School  Primary   
4          3321         Barkisland CofE VA Primary School  Primary   
..          ...                                       ...      ...   
96         2028               Warley Road Primary Academy  Primary   
97         2030                        Warley Town School  Primary   
98         2016                         West Vale Academy  Primary   
99         2046               Withinfields Primary School  Primary   
101        2081                  Woodhouse Primary School  Primary   

                     Status   Head title            Head name  \
0       Academy sponsor led  Headteacher    Ms Natasha Searby   
1    Voluntary aided school  

In [21]:
#project to meters for flat Voronoi grid
gdf_m = gdf.to_crs("EPSG:27700")

#new projected coordinates
coords = np.column_stack((gdf_m.geometry.x, gdf_m.geometry.y))


In [22]:
#create Voronoi grid by passng 'longitude' and 'latitude' from gdf to Voronoi class

#vor = Voronoi(gdf[['Longitude', 'Latitude']])
vor = Voronoi(coords) #using coorindates of a flat map instead

#print (gdf[['Longitude', 'Latitude']])


In [23]:
#make list of Voronoi polygons and remove invalid regions (empty or extend to infinity)
##if 'region' attribute of vor (from Voronoi() class) starts wth '-1' the regon never closes
##pass vertices attribute to shapely's Polygon() class. Generates plottable polygons
voronoi_polygons = [Polygon(vor.vertices[region]) for region in vor.regions if region and -1 not in region]

#create GeoDataFrame with Voronoi polygons
#gdf_voronoi = gpd.GeoDataFrame(geometry = voronoi_polygons, crs='EPSG:4326')

gdf_m_voronoi = gpd.GeoDataFrame(geometry=voronoi_polygons, crs='EPSG:27700')


In [24]:
#Apply truncation to limit area to within Calderdale Area

# Define the bounding box lat-lon limits: - ALTER MAX AND MIN LAT AND LONG VALS AS REQUIRED
##max_lat, min_lat, max_lon, min_lon = (53.9, 53.5, -1.8, -2.25)

min_x, min_y, max_x, max_y = gdf_m.total_bounds #create bounds from data vals instead of lat and long vals
pad = 5000 #5k distance as padding for bounds

# Create the bounding box as a Shapely Polygon
##bounding_box = Polygon.from_bounds(min_lon, min_lat, max_lon, max_lat)
bounding_box_m = Polygon.from_bounds(min_x - pad, min_y - pad, max_x + pad, max_y + pad)

# Truncate each Voronoi polygon with the bounding box:
##truncated_polygons = [polygon.intersection(bounding_box) for polygon in gdf_voronoi.geometry]
truncated_polygons_m = [polygon.intersection(bounding_box_m) for polygon in gdf_m_voronoi.geometry]


# Create a GeoDataFrame with the truncated polygons:
##gdf_truncated = gpd.GeoDataFrame(geometry=truncated_polygons, crs='EPSG:4326')
gdf_m_truncated = gpd.GeoDataFrame(geometry=truncated_polygons_m,crs='EPSG:27700')

#convert polygons back to longitude and latitude
gdf_m_truncated = gdf_m_truncated.to_crs('EPSG:4326')



In [25]:
#Plot map - using Folium to draw it over a street map of the city
## -> choose openstreetmap as the tiles argument
## -> control icon inside marker by setting the icon symbol

#folium map centred on average coordinates of the schools:
map_centre = [gdf['Latitude'].mean(), gdf['Longitude'].mean()]
school_map = folium.Map(location = map_centre, zoom_start =12, tiles = 'OpenStreetmap')

#plot Voronoi polygons on the map
folium.GeoJson(gdf_m_truncated).add_to(school_map)

#add markers for each school
for index, school in gdf.iterrows():
  folium.Marker(location = [school['Latitude'], school['Longitude']], popup = f"{school['Establishment']}\n{school['Postcode']}\n{school['Address 4']}", icon=folium.Icon(color='blue', icon='home')).add_to(school_map)

#display map
school_map #displays primary and secondary map

In [ ]:
##To further develop: can blend other limitations/ incporporate other boundaries i.e. closest transport-wise instead of purely nearest neighbour into the determining of the Voronoi Diagrams